In [4]:
# %% [markdown]
# # Iconify → Power BI Local CSV Library Builder

# %% Setup
import requests
import csv
import time
import os

API_BASE = "https://api.iconify.design"
OUTPUT_DIR = "iconify_library"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# %% Functions
def get_all_sets() -> dict:
    """List all available icon sets."""
    resp = requests.get(f"{API_BASE}/collections")
    resp.raise_for_status()
    return resp.json()


def get_icon_names(prefix: str) -> list:
    """Get all icon names in a set."""
    resp = requests.get(f"{API_BASE}/collection", params={"prefix": prefix})
    resp.raise_for_status()
    data = resp.json()

    icons = []
    if "uncategorized" in data:
        icons.extend(data["uncategorized"])
    if "categories" in data:
        for cat_icons in data["categories"].values():
            icons.extend(cat_icons)
    return sorted(set(icons))


def get_svg(prefix: str, name: str, height: int = 24) -> str | None:
    """Fetch a single SVG from the API."""
    try:
        resp = requests.get(
            f"{API_BASE}/{prefix}/{name}.svg",
            params={"height": height}
        )
        if resp.status_code == 200:
            return resp.text
    except Exception:
        pass
    return None


def svg_to_data_uri(svg: str) -> str:
    """Convert raw SVG to a Power BI-ready data URI."""
    cleaned = svg.replace('"', "'")
    return f"data:image/svg+xml;utf8, {cleaned}"


def fetch_and_save_set(prefix: str, overwrite: bool = False):
    """Fetch all icons in a set and save to CSV."""
    output_path = os.path.join(OUTPUT_DIR, f"{prefix}.csv")

    if os.path.exists(output_path) and not overwrite:
        print(f"[{prefix}] Already exists — skipping (set overwrite=True to re-fetch)")
        return

    print(f"[{prefix}] Fetching icon list...")
    icons = get_icon_names(prefix)
    print(f"[{prefix}] Found {len(icons)} icons. Fetching SVGs...")

    count = 0
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["SetName", "IconName", "FullName", "DataImage"])

        for i, name in enumerate(icons):
            svg = get_svg(prefix, name)
            if svg:
                data_uri = svg_to_data_uri(svg)
                writer.writerow([prefix, name, f"{prefix}:{name}", data_uri])
                count += 1

            if (i + 1) % 50 == 0:
                print(f"  [{prefix}] {i + 1}/{len(icons)}")

            if (i + 1) % 25 == 0:
                time.sleep(0.5)

    print(f"[{prefix}] Done! Saved {count} icons → {output_path}\n")


# %% Browse available sets (optional — run this to see what's out there)
sets = get_all_sets()
print(f"Available icon sets: {len(sets)}\n")
print(f"{'Prefix':<25} {'Name':<40} {'Icons':>6}")
print(f"{'-'*25} {'-'*40} {'-'*6}")
for prefix, info in sorted(sets.items(), key=lambda x: x[1].get("name", "")):
    print(f"{prefix:<25} {info.get('name',''):<40} {info.get('total',0):>6}")


# %% FETCH ICONS — edit this list and run
sets_to_fetch = ["cbi","bi","clarity"]

print(f"Fetching {len(sets_to_fetch)} sets → ./{OUTPUT_DIR}/\n")
for prefix in sets_to_fetch:
    fetch_and_save_set(prefix)

print("=" * 50)
print(f"Library saved to ./{OUTPUT_DIR}/")
print("Point Power Query at this folder.")


# %% Fetch additional sets later (just add to the list and run this cell)
# extra_sets = ["solar", "heroicons", "bi"]
# for prefix in extra_sets:
#     fetch_and_save_set(prefix)

Available icon sets: 225

Prefix                    Name                                      Icons
------------------------- ---------------------------------------- ------
academicons               Academicons                                 158
akar-icons                Akar Icons                                  454
ant-design                Ant Design Icons                            830
arcticons                 Arcticons                                 14559
bpmn                      BPMN                                        112
basil                     Basil                                       493
bitcoin-icons             Bitcoin Icons                               250
bi                        Bootstrap Icons                            2078
bx                        BoxIcons v2                                 814
bxs                       BoxIcons v2 Solid                           665
boxicons                  Boxicons                                   3768
bxl         